In [1]:
import sys

sys.path.append("..")

from matchms.importing import load_from_msp
from tqdm import tqdm

from SpecEmbedding.utils.clean import (
    apply_filters,
    clean_metadata,
    clean_metadata2,
    count_annotations,
    filter_by_precursor_mz,
    is_annotated,
    minimal_processing,
    seperate_spectra_by_ionmode,
)

path = "../data/legacy/MassBank.msp_NIST"
spectra = list(load_from_msp(path))

after_filter = [apply_filters(s) for s in tqdm(spectra)]
after_filter = [s for s in tqdm(after_filter) if s is not None]
cleaned = [clean_metadata(s) for s in tqdm(after_filter)]
cleaned = [clean_metadata2(s) for s in tqdm(cleaned)]

positive, negative = seperate_spectra_by_ionmode(cleaned)
positive = [minimal_processing(s) for s in tqdm(positive)]
positive = [s for s in positive if s is not None]
count_annotations(positive, "peak num >= 5")
positive = filter_by_precursor_mz(positive)
count_annotations(positive, "10 < precursor_mz < 1000")
annotated = is_annotated(positive)
count_annotations(annotated, "annotated")

100%|██████████| 73088/73088 [00:00<00:00, 200238.34it/s]


peak num >= 5
compound_name: 73088 -- unique: 18992
inchi: 73088 -- unique: 15551
smiles: 73088 -- unique: 18410
inchikey: 73088 -- unique: 14058


100%|██████████| 52422/52422 [00:00<00:00, 219869.41it/s]


10 < precursor_mz < 1000
compound_name: 52422 -- unique: 6930
inchi: 52422 -- unique: 6084
smiles: 52422 -- unique: 8346
inchikey: 52422 -- unique: 5289


100%|██████████| 52137/52137 [00:00<00:00, 208192.40it/s]

annotated
compound_name: 52137 -- unique: 6799
inchi: 52137 -- unique: 6083
smiles: 52137 -- unique: 8345
inchikey: 52137 -- unique: 5288


In [2]:
from collections import defaultdict

from tqdm import tqdm

instrument2spectra = defaultdict(list)

instruments = []

for s in tqdm(annotated):
    instrument: str = s.metadata.get("instrument", None)
    if instrument is None:
        instrument = s.metadata.get("instrument_type")

    if "orbitrap" in instrument.lower():
        instrument2spectra["Orbitrap"].append(s)
    elif "qtof" in instrument.lower() or "q-tof" in instrument.lower() or "tof" in instrument.lower():
        instrument2spectra["QTOF"].append(s)
    else:
        instruments.append(instrument)

instrument2spectra["all"] = instrument2spectra["Orbitrap"] + instrument2spectra["QTOF"]
instrument2spectra = dict(instrument2spectra)

100%|██████████| 52137/52137 [00:00<00:00, 351788.52it/s]


In [4]:
from pathlib import Path

import numpy as np

from SpecEmbedding.utils.clean import get_ref_query, get_unique_smiles

path_dir = Path("../data/legacy/MassBank")
path_dir.mkdir(parents=True, exist_ok=True)

replica_suffix = "-replication-{}"

train_ref_spectra = np.load("../data/legacy/MSBert/GNPS/Orbitrap/train_ref.npy", allow_pickle=True)
train_ref_smiles = get_unique_smiles(train_ref_spectra)

for instrument_type, spectra in instrument2spectra.items():
    print(instrument_type)
    smiles_seq = get_unique_smiles(spectra)
    print(len(smiles_seq))
    bool_indices = np.isin(smiles_seq, train_ref_smiles)
    exclued_smiles = smiles_seq[~bool_indices]
    print(len(exclued_smiles))
    for i in range(10):
        query, reference = get_ref_query(spectra, exclued_smiles)
        np.save(path_dir.joinpath(instrument_type + "-query" + replica_suffix.format(i + 1)), query)
        np.save(path_dir.joinpath(instrument_type + "-reference" + replica_suffix.format(i + 1)), reference)

Orbitrap
2998
2637


split query and reference set: 100%|██████████| 2637/2637 [00:00<00:00, 97041.30it/s]


QTOF
3941
3705


split query and reference set: 100%|██████████| 3705/3705 [00:00<00:00, 147103.78it/s]


all
6666
6116


split query and reference set: 100%|██████████| 6116/6116 [00:00<00:00, 128186.83it/s]
